# Train the optical-SAR fusion classifier (Stage 2)

**Goal:** Stage 1 of `optical_sar_fusion` (`models/fusion/fusion_tool.py`) is training-free -- recursive
Otsu thresholding on SAR backscatter physics, cross-checked against the existing water-segmentation
tool's optical read. It has **no trained model at all**, and its built-up (not just water) read is
"SAR-only... no optical cross-check exists yet" by the adapter's own documented admission -- of every
mandatory capability in the spec, this is the one with the least real machine learning behind it.
This notebook trains a small **early-fusion CNN** that looks at a co-registered optical+SAR patch
*together* and predicts which of four broad land-cover types it shows (**agricultural, barren,
grassland, urban**), giving a second, learned, genuinely cross-modal signal -- in particular a
learned P(urban) to actually cross-check Stage 1's SAR-only built-up read, closing that documented gap.

**Data:** [Sentinel-1&2 Image Pairs (SAR & Optical)](https://www.kaggle.com/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain)
(CC BY 4.0), via the Kaggle copy `requiemonk/sentinel12-image-pairs-segregated-by-terrain` -- co-registered
Sentinel-1 (SAR) / Sentinel-2 (optical) 256x256 patches from the TUM SEN1-2 dataset, curated into four
land-cover classes. Attached as a notebook input, no download step.

## What was checked before writing this (so the cells below aren't guesses)

* **Layout**, confirmed via the dataset's own file browser and the Kaggle API: `v_2/{agri,barrenland,
  grassland,urban}/{s1,s2}/ROIs<collection>_<season>_s<1|2>_<scene>_p<patch>.png` -- s1 (SAR) and s2
  (optical) filenames for the same patch differ only in the `s1`/`s2` token, so pairs join by name.
  **4,000 files per class per modality** (16,000 pairs total, ~2.7GB), confirmed both from the
  dataset's own UI ("4000 files" on the grassland/s1 and /s2 folders) and independently from the total
  download size (2,734,660,827 bytes / ~171KB per pair matches).
* **Channels and size**, checked by opening eight real files (two per class) directly: SAR (s1) is
  single-channel 8-bit grayscale ('L' mode), optical (s2) is 3-channel 8-bit RGB, both exactly 256x256
  -- matches `fusion_tool.py`'s existing `Image.open(sar_path).convert("L")` assumption.
  * **Class content, checked by rendering and looking at real pairs from all four classes** (the
    lesson of this project's own mislabelled-test-image mistakes): agri shows real green/brown field
    patchwork; barrenland shows bare rock and a mountain lake; grassland shows green terrain with a
    lake and a small structure (matches the dataset's own description -- "grasslands also show some
    geographical features, with an occasional river"); urban's *first* sampled patch looked like mostly
    farmland with a small building cluster, which briefly looked like a labelling problem -- **checking
    four more urban patches spanning the full patch-number range showed dense, unambiguous built-up
    texture in the optical image and a correspondingly bright, rough SAR signature** in every one, so
    the first patch was just an edge-of-scene tile, not evidence the class is mislabelled.
* **Each class is drawn from few real source scenes, not many -- but not always exactly one either,
  which a real run caught the hard way.** Every one of ~10 manually sampled filenames per class
  shared one `<scene>` id (urban = scene 13; barrenland = 114; grassland = 115) and three probes for
  other plausible scene ids under `urban/s1` all came back not-found, so the notebook was first
  written assuming one scene per class. **The first real Kaggle run's own leak-check assertion fired
  immediately** ("agri: train/val patch ids overlap") -- turns out at least the agri class spans more
  than one source scene, with patch numbers restarting per scene, so a split keyed on the bare patch
  number alone can collide two physically different tiles onto both sides of the split. Fixed by
  keying the split on **(scene id, patch id)** together, not the patch number alone -- see the next
  section. **This means a plain random train/val split by patch would leak regardless** -- nearby
  patches of the same real place would land on both sides, testing "does it recognize this specific
  place" rather than "does it generalize to a new one." The split below is deliberate, not random.

## Design

* **Early-fusion CNN**: optical (3ch) and SAR (1ch) stacked into one 4-channel input, fed to a
  ResNet18 whose first conv is expanded from 3 to 4 input channels (the pretrained RGB weights copied
  into channels 0-2, the new SAR channel initialized as their mean -- a standard, well-known channel-
  expansion transfer trick) -- everything else keeps its ImageNet pretraining.
* **Two baselines trained the same way**: optical-only (3ch, unmodified ResNet18) and SAR-only (1ch,
  first conv averaged to one channel) -- so the notebook *measures* whether fusion actually beats
  either modality alone, rather than assuming it, matching this project's own comparison discipline
  elsewhere (zero-shot vs fine-tuned, Stage 1 vs Stage 2).
* **Split**: within each class, patches are sorted by their **(scene id, patch id)** key and the
  highest ~15% held out for validation -- deterministic, not random. When a class turns out to span
  more than one scene (see above), this also tends to hold out whichever scene sorts last as a whole,
  a better approximation of "a new place" than a same-scene patch split; when a class really is one
  scene, it degrades to that within-scene generalization check. Documented as such, not oversold as a
  to-a-genuinely-new-place test in every case -- the notebook prints each class's distinct scene count
  so this isn't left as an assumption either.
* **Evaluation**: accuracy, confusion matrix, per-class precision/recall for all three models
  (fusion / optical-only / SAR-only) side by side, plus specifically **P(urban) as a built-up signal**
  compared against Stage 1's own SAR-only builtup_fraction on a few held-out patches.

Run cells top to bottom. Kaggle: enable **Internet** (ImageNet weights) and a **GPU**, and attach the
SEN1-2 dataset as an input.

## 0. GPU compatibility check

**Found live, via an actual failed run on Kaggle**: this session's preinstalled PyTorch build
(2.10.0+cu128) only supports CUDA compute capabilities sm_70 and up -- it silently drops support
for the Pascal-generation **P100** (sm_60), one of the two GPU types Kaggle itself still offers
here (T4x2 or P100, per the note below). Landing on a P100 crashed training ~40 seconds in with
`CUDA error: no kernel image is available for execution on the device` -- a real run, not a
hypothetical. The cell below detects the actual GPU via `nvidia-smi` (no torch import needed yet,
so this runs before torch's own compute-capability list is fixed for the process) and reinstalls
a CUDA 11.8 build if the assigned GPU isn't in the preinstalled build's supported list -- CUDA 11.8
wheels cover Pascal through Hopper, so this works regardless of which GPU Kaggle happens to assign.

In [ ]:
import subprocess, sys

try:
    cc_raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc_raw.split(".")
    needed_sm = f"sm_{major}{minor}"
except Exception as e:
    needed_sm = None
    print(f"Could not query GPU compute capability via nvidia-smi ({e}) -- skipping the compatibility check.")

if needed_sm:
    # Check the INSTALLED build's supported architectures in a SEPARATE PROCESS, not an in-process
    # `import torch` -- Python caches imports in sys.modules, so even an aliased/deleted in-process
    # import here would make a LATER `import torch` in the next cell silently return the stale
    # cached module instead of a fresh one. Confirmed live: an earlier version of this cell did
    # `import torch as _torch_probe`, and the kernel died ~80s after reinstalling -- the old torch
    # stayed resident in this process while its .so files got replaced out from under it on disk.
    check = subprocess.run(
        [sys.executable, "-c",
         "import torch; print(' '.join(torch.cuda.get_arch_list()) if torch.cuda.is_available() else '')"],
        capture_output=True, text=True,
    )
    supported = check.stdout.split()
    if needed_sm not in supported:
        print(f"GPU needs {needed_sm}, not in the preinstalled torch build's supported list "
              f"{supported} -- reinstalling a CUDA 11.8 build (covers Pascal through Hopper)...")
        # Uninstall the whole torch/torchvision/torchaudio trio first, then install all three
        # together from the SAME cu118 index in one resolution -- reinstalling `torch` alone
        # left mismatched torchvision/nccl versions behind, which crashed two live runs with
        # unrelated-looking errors (undefined symbol ncclCommShrink; aten.OpaqueObject not
        # registered) that were actually both this same root cause from a different angle.
        subprocess.run(["pip", "uninstall", "-y", "-q", "torch", "torchvision", "torchaudio"], check=True)
        subprocess.run(
            ["pip", "install", "-q", "torch", "torchvision", "torchaudio",
             "--index-url", "https://download.pytorch.org/whl/cu118"],
            check=True,
        )
        print("Reinstalled. The check above ran in a subprocess, so THIS process has never "
              "imported torch itself -- the next cell's `import torch` will be a genuinely fresh "
              "import, not a cached one.")
    else:
        print(f"GPU compute capability {needed_sm} already supported by the preinstalled build.")

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))

## 1. Setup

In [ ]:
import os, sys, json, math, time, random, glob, collections
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision

# Local smoke-test hooks -- never set on Kaggle. SF_DATA_ROOT points at a flat folder with
# {agri,barrenland,grassland,urban}/{s1,s2}/*.png subfolders (a handful of real pairs per class) so
# the whole notebook can run end to end on a laptop before any GPU time is spent.
SMOKE_TEST = bool(os.environ.get("SF_SMOKE_TEST"))
DATA_ROOT = os.environ.get("SF_DATA_ROOT")

SEED = 0
CLASSES = ["agri", "barrenland", "grassland", "urban"]
NUM_CLASSES = len(CLASSES)
URBAN_IDX = CLASSES.index("urban")
CROP = 224  # ResNet's native training resolution; patches are 256x256 so this is a light center-safe crop
BATCH = 4 if SMOKE_TEST else 64
EPOCHS = 1 if SMOKE_TEST else 15
WORKERS = 0 if SMOKE_TEST else 4
LR = 3e-4
VAL_FRACTION = 0.15  # highest-numbered 15% of each class's patch ids held out -- see the design note above

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| smoke test:", SMOKE_TEST)

## 2. Find the data and build the deterministic (non-random) split

Pairs are found per class by joining `s1`/`s2` filenames that share everything except the `s1`/`s2`
token, then sorted by their **(scene id, patch id)** key -- the highest `VAL_FRACTION` of each
class's keys become validation, the rest train. This is deliberate (see the design note above), not
`random.shuffle`. The key is the *pair*, not the bare patch number: an earlier version keyed on patch
id alone and a real run caught its own bug immediately (the sanity check below fired) -- "agri" turns
out to draw from more than one source scene, and patch numbers restart per scene, so two physically
different tiles can share the same bare patch id. Keying on (scene, patch) both fixes the crash (the
pair is unique) and, when a class does span multiple scenes, tends to hold out whichever scene sorts
last as validation -- a better approximation of "a new place," not just "a new tile of the same place."

In [ ]:
import re

PATCH_RE = re.compile(r"_s[12]_(\d+)_p(\d+)\.png$")  # -> (scene_id, patch_id), both as zero-padded strings for a stable sort

def find_root():
    if DATA_ROOT:
        return DATA_ROOT
    for d in glob.glob("/kaggle/input/**/v_2", recursive=True):
        if all(os.path.isdir(f"{d}/{c}/s1") for c in CLASSES):
            return d
    raise AssertionError("SEN1-2 dataset not found under /kaggle/input -- add "
                          "'requiemonk/sentinel12-image-pairs-segregated-by-terrain' as an input")

ROOT = find_root()
print("root:", ROOT)

def class_pairs(cls):
    """((scene_id, patch_id), optical_path, sar_path) for one class, sorted by that key."""
    s2_dir = f"{ROOT}/{cls}/s2"
    out = []
    for s2_path in glob.glob(f"{s2_dir}/*.png"):
        s1_path = s2_path.replace(f"/{cls}/s2/", f"/{cls}/s1/").replace("_s2_", "_s1_")
        m = PATCH_RE.search(s2_path)
        if os.path.exists(s1_path) and m:
            key = (m.group(1).zfill(6), m.group(2).zfill(6))  # zero-pad so string sort == numeric sort
            out.append((key, s2_path, s1_path))
    out.sort(key=lambda t: t[0])
    return out

train_entries, val_entries = [], []
class_counts = {}
for ci, cls in enumerate(CLASSES):
    pairs = class_pairs(cls)
    assert pairs, f"no pairs found for class {cls}"
    n_scenes = len({key[0] for key, _, _ in pairs})
    n_val = max(1, int(round(len(pairs) * VAL_FRACTION)))
    tr, va = pairs[:-n_val], pairs[-n_val:]
    class_counts[cls] = {"total": len(pairs), "train": len(tr), "val": len(va), "distinct_scenes": n_scenes}
    train_entries += [(ci, s2, s1) for _, s2, s1 in tr]
    val_entries += [(ci, s2, s1) for _, s2, s1 in va]
print("pairs per class:", class_counts)
print(f"train {len(train_entries)} | val {len(val_entries)}")

# sanity: no (scene, patch) key appears in both train and val for any class -- the split is a clean
# cut, not scattered. This is the one leakage check that's actually enforceable given the scene-
# scarcity finding: it can't check "different physical place" when a class has only one source scene
# (the data doesn't offer that), only "different tile" -- but it CAN and must catch a key collision.
for cls in CLASSES:
    pairs = class_pairs(cls)
    n_val = max(1, int(round(len(pairs) * VAL_FRACTION)))
    train_keys = {key for key, _, _ in pairs[:-n_val]}
    val_keys = {key for key, _, _ in pairs[-n_val:]}
    assert not (train_keys & val_keys), f"{cls}: train/val (scene, patch) keys overlap"
    assert len(train_keys) + len(val_keys) == len(pairs), f"{cls}: duplicate (scene, patch) keys within the class itself"
print("no train/val key overlap and no duplicate keys in any class")

## 3. Dataset, model (early fusion + two single-modality baselines)

Optical gets flips/rot90 plus a mild brightness/contrast/saturation jitter; SAR gets flips/rot90 only
(no photometric jitter -- backscatter intensity is the actual signal, not a rendering choice like roof
colour). Both are resized 256->224 the same way for every model so the three are comparable.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
# SAR's own per-channel mean/std, computed once (cell 5) from a real training sample rather than
# reusing an RGB constant that has no reason to fit a single-channel radar-intensity image.
SAR_MEAN, SAR_STD = None, None

def load_pair(optical_path, sar_path):
    optical = np.array(Image.open(optical_path).convert("RGB"), dtype=np.uint8)
    sar = np.array(Image.open(sar_path).convert("L"), dtype=np.uint8)
    return optical, sar

class FusionDataset(Dataset):
    def __init__(self, entries, train: bool):
        self.entries, self.train = entries, train
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, i):
        label, optical_path, sar_path = self.entries[i]
        optical, sar = load_pair(optical_path, sar_path)
        opt = torch.from_numpy(optical).permute(2, 0, 1).float() / 255.0
        sr = torch.from_numpy(sar).float()[None] / 255.0
        if self.train:
            if random.random() < 0.5:
                opt, sr = opt.flip(-1), sr.flip(-1)
            if random.random() < 0.5:
                opt, sr = opt.flip(-2), sr.flip(-2)
            k = random.randint(0, 3)
            if k:
                opt, sr = torch.rot90(opt, k, (-2, -1)), torch.rot90(sr, k, (-2, -1))
            opt = (opt - opt.mean()) * random.uniform(0.9, 1.1) + opt.mean() * random.uniform(0.9, 1.1)
            gray = opt.mean(0, keepdim=True)
            opt = (gray + (opt - gray) * random.uniform(0.85, 1.15)).clamp(0, 1)
        opt = F.interpolate(opt[None], size=(CROP, CROP), mode="bilinear", align_corners=False)[0]
        sr = F.interpolate(sr[None], size=(CROP, CROP), mode="bilinear", align_corners=False)[0]
        opt = (opt - torch.tensor(IMAGENET_MEAN)[:, None, None]) / torch.tensor(IMAGENET_STD)[:, None, None]
        sr = (sr - SAR_MEAN) / SAR_STD
        return opt, sr, label

def make_loader(entries, train, batch=None):
    return DataLoader(FusionDataset(entries, train), batch_size=batch or BATCH, shuffle=train,
                       drop_last=train, num_workers=WORKERS, pin_memory=(DEVICE == "cuda"))

def build_model(in_channels: int):
    """A ResNet18 whose first conv is expanded/contracted to `in_channels`: 4 = fusion (RGB+SAR),
    3 = optical baseline (unmodified), 1 = SAR baseline. The RGB pretrained weights are always kept;
    a channel that isn't in the pretrained 3 is initialized as the mean of those three, the standard
    trick for adapting an ImageNet stem to a different channel count without discarding its features."""
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    if in_channels != 3:
        old = m.conv1.weight.data  # (64, 3, 7, 7)
        new = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        mean_w = old.mean(dim=1, keepdim=True)  # (64, 1, 7, 7)
        if in_channels == 1:
            new.weight.data = mean_w
        else:  # 4 = RGB kept + one averaged extra channel appended
            new.weight.data = torch.cat([old, mean_w], dim=1)
        m.conv1 = new
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

MODEL_INPUTS = {"fusion": lambda opt, sr: torch.cat([opt, sr], dim=1),
                "optical_only": lambda opt, sr: opt,
                "sar_only": lambda opt, sr: sr}

## 4. Metrics

In [ ]:
def confusion(preds, labels):
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for p, l in zip(preds, labels):
        cm[l, p] += 1
    return cm

def metrics_from_confusion(cm):
    tp = np.diag(cm).astype(float)
    pred_pos = cm.sum(0); actual_pos = cm.sum(1)
    precision = np.divide(tp, pred_pos, out=np.zeros_like(tp), where=pred_pos > 0)
    recall = np.divide(tp, actual_pos, out=np.zeros_like(tp), where=actual_pos > 0)
    return {
        "accuracy": float(tp.sum() / max(cm.sum(), 1)),
        "per_class": {CLASSES[c]: {"precision": float(precision[c]), "recall": float(recall[c]),
                                    "support": int(actual_pos[c])} for c in range(NUM_CLASSES)},
        "confusion_matrix": cm.tolist(),
    }

@torch.no_grad()
def evaluate(model, kind, loader):
    model.eval()
    preds, labels, urban_probs = [], [], []
    for opt, sr, y in loader:
        opt, sr, y = opt.to(DEVICE), sr.to(DEVICE), y.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            logits = model(MODEL_INPUTS[kind](opt, sr))
        probs = F.softmax(logits.float(), dim=1)
        preds += probs.argmax(1).cpu().tolist()
        labels += y.cpu().tolist()
        urban_probs += probs[:, URBAN_IDX].cpu().tolist()
    cm = confusion(preds, labels)
    out = metrics_from_confusion(cm)
    out["mean_urban_probability_on_urban_patches"] = float(np.mean([p for p, l in zip(urban_probs, labels) if l == URBAN_IDX])) if URBAN_IDX in labels else None
    out["mean_urban_probability_on_other_patches"] = float(np.mean([p for p, l in zip(urban_probs, labels) if l != URBAN_IDX])) if any(l != URBAN_IDX for l in labels) else None
    return out

## 5. Pre-flight: SAR normalization stats, then the whole train/save/reload path on synthetic data

Real per-channel mean/std for SAR are computed from a sample of real training patches first (needed
before any dataset item can be constructed, since `FusionDataset.__getitem__` normalizes with it) --
then the usual pre-flight (model build for all three input widths, fp16 steps, save/reload) runs on
synthetic tensors before any real data or GPU time is spent training.

In [ ]:
def compute_sar_stats(entries, n=200):
    sample = random.Random(SEED).sample(entries, min(n, len(entries)))
    vals = []
    for _, _, sar_path in sample:
        vals.append(np.asarray(Image.open(sar_path).convert("L"), dtype=np.float32).ravel() / 255.0)
    vals = np.concatenate(vals)
    return float(vals.mean()), float(max(vals.std(), 1e-6))

sar_mean, sar_std = compute_sar_stats(train_entries)
SAR_MEAN, SAR_STD = torch.tensor([sar_mean]), torch.tensor([sar_std])
print(f"SAR normalization from {min(200, len(train_entries))} real training patches: mean {sar_mean:.4f} std {sar_std:.4f} (ImageNet RGB uses ~0.45/0.22 -- SAR is its own distribution)")

def preflight():
    t0 = time.time()
    use_amp = DEVICE == "cuda"
    for kind, ch in (("fusion", 4), ("optical_only", 3), ("sar_only", 1)):
        m = build_model(ch).to(DEVICE)
        opt_optim = torch.optim.AdamW(m.parameters(), lr=1e-4)
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        opt = torch.randn(2, 3, CROP, CROP, device=DEVICE)
        sr = torch.randn(2, 1, CROP, CROP, device=DEVICE)
        y = torch.randint(0, NUM_CLASSES, (2,), device=DEVICE)
        m.train()
        for _ in range(2):
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(m(MODEL_INPUTS[kind](opt, sr)), y)
            assert torch.isfinite(loss), f"[{kind}] non-finite loss ({float(loss)})"
            scaler.scale(loss).backward(); scaler.unscale_(opt_optim)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt_optim); scaler.update(); opt_optim.zero_grad(set_to_none=True)
        path = "/tmp/preflight_fusion.pt"
        torch.save({"model_state_dict": m.state_dict()}, path)
        m2 = build_model(ch); m2.load_state_dict(torch.load(path, map_location="cpu")["model_state_dict"]); os.remove(path)
        del m, m2, opt_optim
        if DEVICE == "cuda": torch.cuda.empty_cache()
        print(f"  [{kind}] build/fp16-step/save/reload OK ({ch}ch input)")
    print(f"pre-flight OK in {time.time() - t0:.0f}s")

preflight()

def pick_batch(target):
    if DEVICE != "cuda":
        return target
    m = build_model(4).to(DEVICE); m.train()
    scaler = torch.amp.GradScaler("cuda")
    total = torch.cuda.get_device_properties(0).total_memory
    chosen = None
    for cand in [c for c in (256, 128, 96, 64, 32, 16, 8) if c <= max(target, 8)]:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        try:
            opt = torch.randn(cand, 3, CROP, CROP, device=DEVICE)
            sr = torch.randn(cand, 1, CROP, CROP, device=DEVICE)
            y = torch.randint(0, NUM_CLASSES, (cand,), device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(m(torch.cat([opt, sr], dim=1)), y)
            scaler.scale(loss).backward()
            peak = torch.cuda.max_memory_allocated()
        except torch.cuda.OutOfMemoryError:
            peak = total
        for p in m.parameters(): p.grad = None
        opt = sr = y = loss = None
        print(f"  batch {cand:3d}: peak {peak / 1e9:5.1f} GB of {total / 1e9:.1f} GB -> {'fits' if peak < 0.85 * total else 'too big'}")
        if peak < 0.85 * total:
            chosen = cand; break
    assert chosen, "not even batch 8 fits"
    del m, scaler
    torch.cuda.empty_cache()
    return chosen

BATCH = pick_batch(BATCH)
print("batch size:", BATCH)

## 6. Train all three models (fusion, optical-only, SAR-only)

Same recipe for each so the comparison is fair: AdamW, cosine decay after a short warmup, fp16
autocast. Best-by-validation-accuracy weights are kept per model.

In [ ]:
train_loader = make_loader(train_entries, train=True)
val_loader = make_loader(val_entries, train=False, batch=BATCH * 2)
steps_per_epoch = len(train_loader)
steps_total = EPOCHS * steps_per_epoch
warmup = max(1, int(0.05 * steps_total))

def lr_factor(step):
    if step < warmup:
        return (step + 1) / warmup
    progress = (step - warmup) / max(1, steps_total - warmup)
    return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * progress))

def train_one(kind, in_channels):
    model = build_model(in_channels).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    use_amp = DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best_acc, best_state, skipped, step, history = -1.0, None, 0, 0, []
    t_start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        running, n = 0.0, 0
        for opt, sr, y in train_loader:
            opt, sr, y = opt.to(DEVICE, non_blocking=True), sr.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(model(MODEL_INPUTS[kind](opt, sr)), y)
            if not torch.isfinite(loss):
                skipped += 1
                optimizer.zero_grad(set_to_none=True)
                assert skipped <= max(3, 0.05 * (step + 1)), f"[{kind}] too many non-finite losses"
                continue
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            running += loss.item(); n += 1; step += 1
        val = evaluate(model, kind, val_loader)
        history.append({"epoch": epoch + 1, "train_loss": running / max(n, 1), "val_accuracy": val["accuracy"]})
        marker = ""
        if val["accuracy"] > best_acc:
            best_acc, best_state, marker = val["accuracy"], {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "  <- best"
        print(f"  [{kind}] epoch {epoch + 1}/{EPOCHS}  train loss {running / max(n, 1):.4f}  val acc {val['accuracy']:.4f}{marker}", flush=True)
    print(f"[{kind}] training took {(time.time() - t_start) / 60:.1f} min; best val accuracy {best_acc:.4f}; skipped {skipped} non-finite steps")
    model.load_state_dict(best_state)
    return model, history, best_acc

results, histories, trained = {}, {}, {}
for kind, ch in (("fusion", 4), ("optical_only", 3), ("sar_only", 1)):
    model, history, best_acc = train_one(kind, ch)
    trained[kind] = model
    histories[kind] = history
    results[kind] = evaluate(model, kind, val_loader)
    print(f"[{kind}] final validation:", {k: round(v, 4) if isinstance(v, float) else v for k, v in results[kind].items() if k != "confusion_matrix"})

## 7. Compare: does fusion actually beat either modality alone?

In [ ]:
print(f"{'model':14s} {'accuracy':>9s}  " + "  ".join(f"{c:>10s} P/R" for c in CLASSES))
for kind in ("fusion", "optical_only", "sar_only"):
    r = results[kind]
    row = "  ".join(f"{r['per_class'][c]['precision']:.2f}/{r['per_class'][c]['recall']:.2f}   " for c in CLASSES)
    print(f"{kind:14s} {r['accuracy']:9.4f}  {row}")
print(f"\nfusion vs best single modality: {results['fusion']['accuracy']:.4f} vs "
      f"{max(results['optical_only']['accuracy'], results['sar_only']['accuracy']):.4f}")
print(f"\nP(urban) on urban patches vs on other patches (should be well separated if the signal is real):")
for kind in ("fusion", "optical_only", "sar_only"):
    r = results[kind]
    print(f"  [{kind}] on urban: {r['mean_urban_probability_on_urban_patches']:.3f}  |  on others: {r['mean_urban_probability_on_other_patches']:.3f}")

## 8. Export -- the fusion model only (the two baselines exist to justify it, not to ship)

In [ ]:
OUT_DIR = "/tmp/fusion_out" if SMOKE_TEST else "/kaggle/working/fusion_out"
os.makedirs(OUT_DIR, exist_ok=True)
metrics = {
    "classes": CLASSES, "class_counts": class_counts, "val_fraction": VAL_FRACTION,
    "sar_mean": sar_mean, "sar_std": sar_std, "epochs": EPOCHS,
    "fusion": {**results["fusion"], "history": histories["fusion"]},
    "optical_only_baseline": {**results["optical_only"], "history": histories["optical_only"]},
    "sar_only_baseline": {**results["sar_only"], "history": histories["sar_only"]},
    "note": "optical_only_baseline/sar_only_baseline are recorded for comparison only -- not exported as their own checkpoints",
}
torch.save({
    "model_state_dict": trained["fusion"].state_dict(), "classes": CLASSES, "in_channels": 4,
    "crop": CROP, "imagenet_mean": IMAGENET_MEAN, "imagenet_std": IMAGENET_STD,
    "sar_mean": sar_mean, "sar_std": sar_std, "metrics": metrics,
}, os.path.join(OUT_DIR, "fusion_classifier.pt"))
json.dump(metrics, open(os.path.join(OUT_DIR, "fusion_metrics.json"), "w"), indent=1)
print("exported:", sorted(os.listdir(OUT_DIR)), f"({os.path.getsize(os.path.join(OUT_DIR, 'fusion_classifier.pt')) / 1e6:.0f} MB)")

ck = torch.load(os.path.join(OUT_DIR, "fusion_classifier.pt"), map_location="cpu", weights_only=False)
check = build_model(4); check.load_state_dict(ck["model_state_dict"]); check.to(DEVICE); check.eval()
opt0, sr0, y0 = next(iter(val_loader))
with torch.no_grad():
    probs = F.softmax(check(torch.cat([opt0[:1].to(DEVICE), sr0[:1].to(DEVICE)], dim=1)).float(), dim=1)
print("artifact round-trip OK: predicted class", CLASSES[int(probs.argmax(1))], "true class", CLASSES[int(y0[0])], "probs", probs.cpu().numpy().round(3).tolist())